In [1]:
from pathlib import Path
import torch
import flammkuchen as fl
from datetime import datetime
import numpy as np
import string
import sklearn.metrics as metrics
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib qt

import calcium_event_classifier as cec

device = cec.set_device()

# Manual annotation of labels

GUI to enable fast manual annotation of calcium traces.

## Load test dataset

Load the test dataset from an HDF5 file.

In [2]:
# Load test data
test_data_path = Path(r"../datasets/251114_test_dataset.h5")
test_data = fl.load(test_data_path)

# Save output path
today = datetime.now().strftime("%y%m%d")
rater = "ML"
output_path = Path(r"../outputs/{today}_{rater}_manual_annotations.h5".format(today=today, rater=rater))

In [ ]:
cec.run_manual_annotation(data=test_data, save_path=output_path)

## Comparison with labeled test dataset

In [3]:
# Load manual annotation
manual_annotations = fl.load(output_path)

In [ ]:
cm = metrics.confusion_matrix(manual_annotations['label'], test_data['label'])

fig, ax = plt.subplots(figsize=(6, 6))

ax.imshow(cm, cmap=plt.cm.Blues, vmin=0, vmax=np.max(cm))
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted 0', 'Predicted 1'])
ax.set_yticklabels(['0', '1'])

cm_sum = np.sum(cm, axis=1, keepdims=True)
cm_perc = cm / cm_sum.astype(float) * 100
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i,
            f"{cm[i, j]}\n({cm_perc[i, j]:.1f}%)",
            ha='center', va='center',
            color='white' if cm[i, j] > np.max(cm) / 2 else 'black'
        )

ax.set_title("Confusion matrix against labeled data")

recall = metrics.recall_score(manual_annotations['label'], test_data['label'])
precision = metrics.precision_score(manual_annotations['label'], test_data['label'])
print(f"Recall: {recall:.4f}, Precision: {precision:.4f}")

## Compare with model

Define model path variable and load it.

In [4]:
model_path = Path(r"../models/251118_model_dff.pth")
checkpoint = torch.load(model_path, map_location=device)
hyperparams = checkpoint['hyperparams']

C:\Users\dcupolillo\AppData\Local\Temp\ipykernel_28396\3861238806.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device

Initialize classifier with model hyperparameters.

In [5]:
classifier = cec.CalciumEventClassifierDff(
    trace_length =              50,  # Adjust if needed based on your data
    input_channels =            hyperparams["input_channels"],
    conv1_channels =            hyperparams["conv1_channels"],
    conv2_channels =            hyperparams["conv2_channels"],
    conv3_channels =            hyperparams["conv3_channels"],
    conv1_kernel =              hyperparams["conv1_kernel"],
    conv2_kernel =              hyperparams["conv2_kernel"],
    conv3_kernel =              hyperparams["conv3_kernel"],
    leaky_relu_negative_slope = hyperparams["leaky_relu_negative_slope"],
    dropout_rate =              hyperparams["dropout"],
    pool_kernel =               hyperparams["pool_kernel"]
).to(device)

# Load model weights and biases
classifier.load_state_dict(checkpoint['model_state_dict'])
classifier.eval()  # Set to evaluation mode

CalciumEventClassifierDff(
  (res_block1): ResidualBlock(
    (conv): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=same)
    (bn): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): LeakyReLU(negative_slope=0.05)
    (skip_connection): Conv1d(1, 16, kernel_size=(1,), stride=(1,))
    (skip_bn): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (res_block2): ResidualBlock(
    (conv): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=same)
    (bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): LeakyReLU(negative_slope=0.05)
    (skip_connection): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
    (skip_bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (res_block3): ResidualBlock(
    (conv): Conv1d(32, 64, kernel_size=(2,), stride=(1,), padding=same)
    (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, tr

Create test dataset instance for model classification.

In [6]:
test_dataset = cec.DffDataset(
    test_data,
    augment=False,  # No augmentation for test
)

# Create test DataLoader
test_loader = cec.load_test_dataset(
    test_dataset,
    batch_size=hyperparams["batch_size"],
    summary=True,
    shuffle=False
)

----------------------------------

 ==== Test Dataset Summary ====
Batch size: 16
Total samples: 456
Test: 456 samples. → 29 batches
   - Label 0: 245 (53.73%) | Label 1: 211 (46.27%)


Perform classification.

In [7]:
labels, logits = cec.get_predictions_and_labels(
    classifier,
    test_loader,
    device
)

# Convert to numpy arrays and apply sigmoid activation
labels = np.array(labels)
predictions = torch.sigmoid(torch.Tensor(logits)).to("cpu").numpy()

c:\Users\dcupolillo\AppData\Local\anaconda3\envs\2p\lib\site-packages\torch\nn\modules\conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv1d(


Find a threshold that maximizes F1 score, to then binarize the predictions.

In [8]:
thresholds = np.arange(0.0, 1.01, 0.01)

f1_scores = [metrics.f1_score(labels, predictions > t) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best threshold: {best_threshold:.2f} with F1 score: {max(f1_scores):.4f}")

Best threshold: 0.58 with F1 score: 0.9488


Have a look at the results:

In [9]:
fig, ax = plt.subplots(figsize=(6, 6))

cm = metrics.confusion_matrix(labels, predictions > best_threshold)
ax.imshow(cm, cmap=plt.cm.Blues, vmin=0, vmax=np.max(cm))

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted 0', 'Predicted 1'])
ax.set_yticklabels(['0', '1'])

cm_sum = np.sum(cm, axis=1, keepdims=True)
cm_perc = cm / cm_sum.astype(float) * 100
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i,
            f"{cm[i, j]}\n({cm_perc[i, j]:.1f}%)",
            ha='center', va='center',
            color='white' if cm[i, j] > np.max(cm) / 2 else 'black'
        )

ax.set_title("Confusion matrix against model predictions")

recall = metrics.recall_score(labels, predictions > best_threshold)
precision = metrics.precision_score(labels, predictions > best_threshold)
print(f"Recall: {recall:.4f}, Precision: {precision:.4f}")


Recall: 0.9668, Precision: 0.9315


## Build recall matrix

In [10]:
## Build recall matrix: raters + model vs each other

# rows (y-axis) = Rater 1 = reference
# columns (x-axis) = Rater 2 = scorer
# matrix[i, j] = recall of scorer j when rater i is used as reference

rater_paths = list(Path(r"..\outputs").glob("*_manual_annotations.h5"))

scorers = {}
for rater_path in rater_paths:
    rater_name = rater_path.stem.replace("_manual_annotations", "").split("_")[-1]
    scorers[rater_name] = fl.load(rater_path)['label']

scorers["Model"] = (predictions > best_threshold).astype(int)

scorer_names = list(scorers.keys())
n = len(scorer_names)

# Replace rater names with letters, keep "Model" as-is
rater_names = [name for name in scorer_names if name != "Model"]
letter_map = {name: string.ascii_uppercase[i] for i, name in enumerate(rater_names)}
display_names = [letter_map.get(name, name) for name in scorer_names]

# recall_matrix[i, j] = recall of scorer j when i is used as reference
recall_matrix = np.zeros((n, n))

for i, ref_name in enumerate(scorer_names):
    for j, scorer_name in enumerate(scorer_names):
        recall_matrix[i, j] = metrics.recall_score(
            scorers[ref_name], scorers[scorer_name]
        )

# Plot
fig, ax = plt.subplots(figsize=(n * 1.24, n))
cmap = sns.color_palette("rocket", as_cmap=True)
im = ax.imshow(recall_matrix, cmap=cmap, vmin=0, vmax=1)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(display_names)
ax.set_yticklabels(display_names)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{recall_matrix[i, j]:.2f}", ha='center', va='center',
                color='black' if recall_matrix[i, j] > 0.5 else 'white')

# colorbar axis
cbar = plt.colorbar(im, ax=ax, label="Recall")
cbar.ax.set_ylabel("Recall", rotation=-90)
cbar.set_ticks([0, 1.0])

ax_y0 = ax.get_position().y0
ax_height = ax.get_position().height
cbar.ax.set_position(
    [cbar.ax.get_position().x0, ax_y0,
     cbar.ax.get_position().width, ax_height])
